In [4]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [5]:
import torch
import torch.nn as nn
import numpy as np
from language_models.model import RNNModel as lstm




In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
# Dictionary to store activations across tokens
token_gate_activations = {
    'input_gate': [],
    'forget_gate': [],
    'cell_gate': [],
    'output_gate': []
}



In [7]:
def lstm_gate_hook(module, input_args, output):
    """
    Hook function to capture LSTM gate activations at each token.
    
    Args:
        module: The LSTM module
        input_args: Input arguments to the module
        output: Output of the module
    """
    # Only proceed if this is actually an LSTM
    if not isinstance(module, nn.LSTM):
        return
    
    # Get input tensor (x)
    x = input_args[0]
    
    # Get hidden states (h, c) if provided, otherwise use zeros
    if len(input_args) > 1 and input_args[1] is not None:
        h, c = input_args[1]
    else:
        batch_size = x.size(0)
        hidden_size = module.hidden_size
        num_layers = module.num_layers
        num_directions = 2 if module.bidirectional else 1
        
        h = torch.zeros(num_layers * num_directions, batch_size, hidden_size, 
                       device=x.device, dtype=x.dtype)
        c = torch.zeros(num_layers * num_directions, batch_size, hidden_size,
                       device=x.device, dtype=x.dtype)
    
    # We need to manually compute gate activations since they're not exposed
    # Extract weights and biases from the LSTM module
    for layer in range(module.num_layers):
        # Process each token in the sequence
        for token_idx in range(x.size(1)):
            # Get current input (either input at this step or output from prev layer)
            if layer == 0:
                token_input = x[:, token_idx:token_idx+1, :]
            else:
                # For layers after the first, input is output from previous layer
                token_input = layer_outputs[layer-1][:, token_idx:token_idx+1, :]
            
            # Get current hidden state for this layer
            layer_h = h[layer:layer+1]
            layer_c = c[layer:layer+1]
            
            # Extract weights for this layer
            w_ih = getattr(module, f'weight_ih_l{layer}')
            w_hh = getattr(module, f'weight_hh_l{layer}')
            b_ih = getattr(module, f'bias_ih_l{layer}')
            b_hh = getattr(module, f'bias_hh_l{layer}')
            
            # Calculate gate inputs
            # Combine both input-hidden and hidden-hidden contributions
            token_input_flat = token_input.view(token_input.size(0), -1)
            gates = (torch.mm(token_input_flat, w_ih.t()) + b_ih + 
                     torch.mm(layer_h.view(layer_h.size(1), -1), w_hh.t()) + b_hh)
            
            # Split into the four gates
            hidden_size = module.hidden_size
            i, f, g, o = gates.chunk(4, dim=1)
            
            # Apply activations
            i = torch.sigmoid(i)  # input gate
            f = torch.sigmoid(f)  # forget gate
            g = torch.tanh(g)     # cell gate
            o = torch.sigmoid(o)  # output gate
            
            # Store gate activations for this token
            token_gate_activations['input_gate'].append(i.detach().cpu().numpy())
            token_gate_activations['forget_gate'].append(f.detach().cpu().numpy())
            token_gate_activations['cell_gate'].append(g.detach().cpu().numpy())
            token_gate_activations['output_gate'].append(o.detach().cpu().numpy())
            
            # Update cell state
            c = f * c + i * g
            
            # Update hidden state
            h = o * torch.tanh(c)



In [8]:
def load_model(checkpoint_name):
    model = lstm("LSTM", 50001, 650, 650, 2, 0.2, False).to(device)
    with open(checkpoint_name, "rb") as f:
        state_dict = torch.load(f, map_location=device)
        model.load_state_dict(state_dict['model_state_dict'])
    return model

In [10]:
check = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/epoch_40.pt'


In [11]:
model = load_model(check)

In [14]:

model.eval()

# Register the hook to the LSTM layer
for name, module in model.named_modules():
    #if isinstance(module, nn.LSTM):
    module.register_forward_hook(lstm_gate_hook)


In [15]:
model.named_modules

<bound method Module.named_modules of RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 650)
  (rnn): LSTM(650, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)>